In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
import numpy as np

# --- CONFIGURATION ---
GROUND_FILE = 'IQAir_air_quality.csv'
AOD_FOLDER = 'IQAir_stations'
ROOT_OUTPUT = 'Output_TimeSeries_IQAir'

METRICS = ['AQI', 'PM2.5', 'PM10']
METRIC_UNITS = {'AQI': 'Index', 'PM2.5': 'µg/m³', 'PM10': 'µg/m³'}

# Filter thresholds
UNCERTAINTY_THRESHOLDS = [None, 1.5, 1.0, 0.5]

print("Starting IQAir Time Series plotting")

# --- LOAD & CLEAN GROUND DATA ---
df_ground = pd.read_csv(GROUND_FILE)

# Map IQAir columns
column_mapping = {
    'timestamp': 'Timestamp',
    'station_name': 'Name',
    'PM2.5 (µg/m³)': 'PM2.5',
    'PM10 (µg/m³)': 'PM10',
    'aqi': 'AQI'
}
df_ground = df_ground.rename(columns=column_mapping)
df_ground['Timestamp'] = pd.to_datetime(df_ground['Timestamp'])

# Force numeric conversion
for metric in METRICS:
    if metric in df_ground.columns:
        df_ground[metric] = pd.to_numeric(df_ground[metric], errors='coerce')

station_names = df_ground['Name'].unique()

# ---PROCESSING LOOP ---
for station_name in station_names:
    
    # --- FILENAME MATCHING LOGIC ---
    possible_filenames = [
        f"{station_name}.csv",
        f"{station_name.replace(':', '')}.csv",
        f"{station_name.replace(':', '-')}.csv",
        f"{station_name.replace('/', '-')}.csv"
    ]
    
    aod_file_path = None
    for fname in possible_filenames:
        full_path = os.path.join(AOD_FOLDER, fname)
        if os.path.exists(full_path):
            aod_file_path = full_path
            break
            
    if not aod_file_path: continue

    # --- READ AOD DATA ---
    try:
        df_aod_raw = pd.read_csv(aod_file_path)
        df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
        df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])
        df_aod_raw = df_aod_raw.set_index('Timestamp')
    except: continue

    df_station = df_ground[df_ground['Name'] == station_name].copy()
    if df_station.empty: continue
    
    print(f"Processing Station: {station_name}")

    for metric in METRICS:
        if metric not in df_station.columns or df_station[metric].dropna().empty:
            continue

        # Sanitize folder name
        safe_folder_name = "".join([c for c in station_name if c.isalnum() or c in (' ', '-', '_')]).strip()
        save_dir = os.path.join(ROOT_OUTPUT, safe_folder_name, metric)
        os.makedirs(save_dir, exist_ok=True)

        for threshold in UNCERTAINTY_THRESHOLDS:
            
            # --- FILTER LOGIC ---
            df_filter = df_aod_raw.copy()
            
            if 'Uncertainty' in df_filter.columns:
                # Remove Negative Values
                df_filter = df_filter[df_filter['Uncertainty'] >= 0]
                
                # Apply Threshold
                if threshold is not None:
                    df_filter = df_filter[df_filter['Uncertainty'] <= threshold]
            
            df_hourly = df_filter.resample('h').mean().dropna(subset=['AOT'])
            df_hourly = df_hourly.reset_index()
            
            # Merge
            merged = pd.merge(df_station, df_hourly, on='Timestamp', how='inner')
            merged = merged.sort_values('Timestamp')
            
            if len(merged) < 2: continue

            thresh_label = "ALL" if threshold is None else f"U{str(threshold).replace('.', '')}"

            # --- PLOTTING  ---
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, 
                                           gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.1})
            
            # Subplot 1: Main Data
            color_metric = 'tab:blue'
            color_aot = 'tab:orange'
            
            # Ground Data 
            ax1.plot(merged['Timestamp'], merged[metric], color=color_metric, 
                    ms=4, linestyle='-', linewidth=0.8, alpha=0.8,
                     label=f'{metric} (Ground)')
            
            ax1.set_ylabel(f'{metric} ({METRIC_UNITS[metric]})', color=color_metric, fontweight='bold')
            ax1.tick_params(axis='y', labelcolor=color_metric)
            ax1.grid(True, ls='--', alpha=0.5)
            
            # Satellite Data 
            ax1_twin = ax1.twinx()
            ax1_twin.plot(merged['Timestamp'], merged['AOT'], color=color_aot, 
                          ms=6, linestyle='--', linewidth=1, label='AOT (Sat)')
            
            ax1_twin.set_ylabel('AOT', color=color_aot, fontweight='bold')
            ax1_twin.tick_params(axis='y', labelcolor=color_aot)
            
            ax1.set_title(f"Time Series: {metric} vs AOT\n{station_name}\n(Filter: {thresh_label})")
            
            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax1_twin.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

            # Subplot 2: Uncertainty
            if 'Uncertainty' in merged.columns:
                ax2.bar(merged['Timestamp'], merged['Uncertainty'], color='gray', alpha=0.6, width=0.04, label='Uncertainty')
                if threshold: 
                    ax2.axhline(threshold, color='red', ls='--', alpha=0.8, label=f'Threshold ({threshold})')
                    ax2.legend(loc='upper right')
            
            ax2.set_ylabel('Uncertainty', fontweight='bold')
            ax2.set_xlabel('Time')
            ax2.grid(True, ls='--', alpha=0.5)
            ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m\n%Hh'))
            
            filename = f"TimeSeries_{thresh_label}.png"
            plt.savefig(os.path.join(save_dir, filename), dpi=100, bbox_inches='tight')
            plt.close(fig)

print("Done. Check 'Output_TimeSeries_IQAir'.")

Starting IQAir Time Series plotting
Processing Station: Hà Nội: Đại Học Bách Khoa cổng Parabol đường Giải Phóng (KK)
Processing Station: Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Processing Station: Minh Khai - Bắc Từ Liêm
Processing Station: Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK)
Processing Station: Hải Dương: UBND TP. Hải Dương - 106 Đường Trần Hưng Đạo (KK)
Processing Station: Thừa Thiên Huế: 83 đường Hùng Vương (KK)
Processing Station: Thái Bình: Cầu Thái Bình - Đ. Trần Thái Tông - P. Bồ Xuyên - TP Thái Bình (KK)
Processing Station: Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)
Processing Station: Quảng Bình: Khu kinh tế Hòn La (KK)
Processing Station: Trà Vinh: Tp. Trà Vinh (KK)
Processing Station: Hà Nội: Chi cục BVMT (KK)
Processing Station: IQAir Ha Noi
Processing Station: FPT
Processing Station: Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK)
Processing Station: IQAir Vietnam - Saigon Pearl
Processing Station: HCM - FPT Thu

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy import stats 

# --- CONFIGURATION ---
GROUND_FILE = 'IQAir_air_quality.csv'
AOD_FOLDER = 'IQAir_stations'
ROOT_OUTPUT = 'Output_Regression_IQAir'

METRICS = ['AQI', 'PM2.5', 'PM10']
METRIC_UNITS = {'AQI': 'Index', 'PM2.5': 'µg/m³', 'PM10': 'µg/m³'}
UNCERTAINTY_THRESHOLDS = [None, 1.5, 1.0, 0.5]

print("Starting IQAir Regression analysis")

# ---LOAD & CLEAN GROUND DATA ---
df_ground = pd.read_csv(GROUND_FILE)

# Map IQAir columns
column_mapping = {
    'timestamp': 'Timestamp',
    'station_name': 'Name',
    'PM2.5 (µg/m³)': 'PM2.5',
    'PM10 (µg/m³)': 'PM10',
    'aqi': 'AQI'
}
df_ground = df_ground.rename(columns=column_mapping)
df_ground['Timestamp'] = pd.to_datetime(df_ground['Timestamp'])

# Force numeric conversion
for metric in METRICS:
    if metric in df_ground.columns:
        df_ground[metric] = pd.to_numeric(df_ground[metric], errors='coerce')

station_names = df_ground['Name'].unique()

# --- PROCESSING LOOP ---
for station_name in station_names:
    
    # --- FILENAME MATCHING LOGIC ---
    possible_filenames = [
        f"{station_name}.csv",
        f"{station_name.replace(':', '')}.csv",
        f"{station_name.replace(':', '-')}.csv",
        f"{station_name.replace('/', '-')}.csv"
    ]
    
    aod_file_path = None
    for fname in possible_filenames:
        full_path = os.path.join(AOD_FOLDER, fname)
        if os.path.exists(full_path):
            aod_file_path = full_path
            break
            
    if not aod_file_path: continue

    # --- READ AOD DATA ---
    try:
        df_aod_raw = pd.read_csv(aod_file_path)
        df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
        df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])
        df_aod_raw = df_aod_raw.set_index('Timestamp')
    except: continue

    df_station = df_ground[df_ground['Name'] == station_name].copy()
    if df_station.empty: continue
    
    print(f"Processing Regression: {station_name}")

    for metric in METRICS:
        if metric not in df_station.columns or df_station[metric].dropna().empty:
            continue

        # Sanitize folder name
        safe_folder_name = "".join([c for c in station_name if c.isalnum() or c in (' ', '-', '_')]).strip()
        save_dir = os.path.join(ROOT_OUTPUT, safe_folder_name, metric)
        os.makedirs(save_dir, exist_ok=True)

        for threshold in UNCERTAINTY_THRESHOLDS:
            
            # --- FILTER LOGIC ---
            df_filter = df_aod_raw.copy()
            
            if 'Uncertainty' in df_filter.columns:
                # Remove Negative Values
                df_filter = df_filter[df_filter['Uncertainty'] >= 0]
                
                # Apply Threshold
                if threshold is not None:
                    df_filter = df_filter[df_filter['Uncertainty'] <= threshold]
            
            df_hourly = df_filter.resample('h').mean().dropna(subset=['AOT'])
            df_hourly = df_hourly.reset_index()
            
            # Merge
            merged = pd.merge(df_station, df_hourly, on='Timestamp', how='inner')
            
            if len(merged) < 5: continue
            
            thresh_label = "ALL" if threshold is None else f"U{str(threshold).replace('.', '')}"

            # Filter valid pairs
            valid_data = merged[[metric, 'AOT']].dropna()
            
            if len(valid_data) < 5: continue
            
            x = valid_data['AOT'].astype(float)
            y = valid_data[metric].astype(float)
            
            # Regression stats
            slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
            
            # --- PLOTTING ---
            plt.figure(figsize=(7, 6))
            plt.scatter(x, y, alpha=0.6, edgecolors='w', s=50, label='Data points')
            
            # Fit line
            line_x = np.array([x.min(), x.max()])
            plt.plot(line_x, slope * line_x + intercept, 'r-', lw=2, label='Fit Line')
            
            plt.title(f"Regression: {metric} vs AOT\n{station_name}\n(Filter: {thresh_label})")
            plt.xlabel("AOT (Satellite)")
            plt.ylabel(f"{metric} (Ground - {METRIC_UNITS[metric]})")
            plt.grid(True, ls='--', alpha=0.5)
            
            # Stats Text
            stats_text = '\n'.join((
                f'R = {r_value:.2f}',
                f'R² = {r_value**2:.2f}',
                f'N = {len(valid_data)}',
                f'y = {slope:.2f}x + {intercept:.2f}'
            ))
            plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
                           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
            
            filename = f"Regression_{thresh_label}.png"
            plt.savefig(os.path.join(save_dir, filename), dpi=100)
            plt.close()

print("Regression analysis complete. Check 'Output_Regression_IQAir'.")

Starting IQAir Regression analysis
Processing Regression: Hà Nội: Đại Học Bách Khoa cổng Parabol đường Giải Phóng (KK)
Processing Regression: Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)
Processing Regression: Minh Khai - Bắc Từ Liêm
Processing Regression: Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK)
Processing Regression: Hải Dương: UBND TP. Hải Dương - 106 Đường Trần Hưng Đạo (KK)
Processing Regression: Thừa Thiên Huế: 83 đường Hùng Vương (KK)
Processing Regression: Thái Bình: Cầu Thái Bình - Đ. Trần Thái Tông - P. Bồ Xuyên - TP Thái Bình (KK)
Processing Regression: Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)
Processing Regression: Quảng Bình: Khu kinh tế Hòn La (KK)
Processing Regression: Trà Vinh: Tp. Trà Vinh (KK)
Processing Regression: Hà Nội: Chi cục BVMT (KK)
Processing Regression: IQAir Ha Noi
Processing Regression: FPT
Processing Regression: Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK)
Processing Regression: IQAir Vietnam - Sa